# General n-gram regeneration lab

This notebook turns the current theorem-1 regeneration setup into a more general lab.

You can vary:
- `V`: vocabulary size for synthetic urtexts
- `M`: text length of the corpus regenerated each generation
- `alpha`: fraction of the text replaced each generation
- `n = 1, 2, ..., N`: maximum backoff order used to regenerate text
- urtext source: synthetic iid, synthetic latent n-gram support, or an author corpus such as Jane Austen
- optional theorem-2-style lookahead selection over `m`-gram extensions

The run is checkpointed, resumable, and writes versioned figures/metrics to disk.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == "Drift_and_selection":
            return p
        if (p / "GitHub").exists() and (p / "Nat_Paper").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "GitHub" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from drift_selection.ngram_regeneration_lab import (
    AgentConfig,
    LatentGrammarConfig,
    RegenerationLabConfig,
    SelectionConfig,
    UrtextConfig,
    default_output_root,
    run_ngram_regeneration_lab,
)
from drift_selection.theorem1_vocab_drift import CORPUS_REGISTRY

pd.set_option("display.max_columns", 200)


## Configuration

Notes:
- For `URTEXT_MODE = "author_corpus"`, `V` is ignored and `VOCAB_CAP` optionally truncates the observed vocabulary.
- `ORDER_KEEP_PROBS` matters only for `synthetic_latent` urtexts.
- `SELECTION_MODE = "none"` gives the plain theorem-1 regeneration loop.
- `SELECTION_MODE = "reference_rgram"` uses `m`-grams from the urtext as desirable windows, with unseen windows either neutral or undesirable.
- `SELECTION_MODE = "reference_frequency_rgram"` scores windows by how often they appear in the urtext.
- `SELECTION_MODE = "hash_rgram"` creates dense desirable/undesirable `m`-gram families without storing huge sets explicitly.
- `AGENT_MODE` controls how many continuations are explored before deciding what to publish.

In [ ]:
EXPERIMENT_VERSION = "V0_01"
SEED = 123

URTEXT_MODE = "synthetic_latent"  # synthetic_iid | synthetic_latent | author_corpus
AUTHOR_KEY = "jane_austen"        # used only when URTEXT_MODE == "author_corpus"

V = 100
M = 1000
ALPHA = 0.25
GENERATIONS = 20
MAX_ORDER = 3
RESTART_PROBABILITY = 0.0

SOURCE_SLICE = "head"             # head | random_block for author corpora
VOCAB_CAP = None                   # e.g. 500 to compress large author vocabularies

ORDER_KEEP_PROBS = {2: 0.50, 3: 0.25}
EXACT_SUPPORT = False

SELECTION_MODE = "none"          # none | hash_rgram | reference_rgram | reference_frequency_rgram
UTILITY_SPAN = 5
LOOKAHEAD_SAMPLES = 64
DESIRABLE_PROB = 0.25
UNDESIRABLE_PROB = 0.25
REFERENCE_MIN_COUNT = 1
REFERENCE_MAX_PATTERNS = None
REFERENCE_UNSEEN_CATEGORY = "neutral"   # neutral | undesirable
REFERENCE_UNSEEN_SCORE = 0.0              # used for reference_frequency_rgram

AGENT_MODE = "sampled_lookahead"        # sampled_lookahead | tree_search
PUBLISH_STRATEGY = "desirable_then_random"  # desirable_then_random | best_score | softmax_score
LOOKAHEAD_DEPTH = None
CANDIDATE_TRIALS = LOOKAHEAD_SAMPLES
BRANCH_FACTOR = 6
MAX_EXPANSIONS = 2500
PUBLISH_HORIZON = 1
UTILITY_WEIGHT = 1.0
LOGPROB_WEIGHT = 0.0
ROLLBACK_PENALTY = 0.0
TEMPERATURE = 1.0

urtext_tag = AUTHOR_KEY if URTEXT_MODE == "author_corpus" else f"v{V}"
selection_tag = SELECTION_MODE if SELECTION_MODE != "none" else "plain"
RUN_NAME = (
    f"ngram_lab_{URTEXT_MODE}_{urtext_tag}_m{M}_a{str(ALPHA).replace('.', 'p')}_"
    f"n{MAX_ORDER}_{selection_tag}"
)

OUTPUT_ROOT = default_output_root(PROJECT_ROOT)

urtext_cfg = UrtextConfig(
    mode=URTEXT_MODE,
    vocab_size=V,
    author_key=AUTHOR_KEY if URTEXT_MODE == "author_corpus" else None,
    vocab_cap=VOCAB_CAP,
    lowercase=True,
    source_slice=SOURCE_SLICE,
)

latent_cfg = LatentGrammarConfig(
    max_order=MAX_ORDER,
    order_keep_probs=ORDER_KEEP_PROBS,
    exact_support=EXACT_SUPPORT,
    support_cache_limit=4096,
)

lab_cfg = RegenerationLabConfig(
    text_length=M,
    generations=GENERATIONS,
    alpha=ALPHA,
    max_order=MAX_ORDER,
    restart_probability=RESTART_PROBABILITY,
    sample_retained_block=True,
)

selection_cfg = None
if SELECTION_MODE != "none":
    selection_cfg = SelectionConfig(
        mode=SELECTION_MODE,
        span=UTILITY_SPAN,
        lookahead_samples=LOOKAHEAD_SAMPLES,
        desirable_prob=DESIRABLE_PROB,
        undesirable_prob=UNDESIRABLE_PROB,
        reference_min_count=REFERENCE_MIN_COUNT,
        reference_max_patterns=REFERENCE_MAX_PATTERNS,
        reference_unseen_category=REFERENCE_UNSEEN_CATEGORY,
        reference_unseen_score=REFERENCE_UNSEEN_SCORE,
    )

agent_cfg = None
if selection_cfg is not None:
    agent_cfg = AgentConfig(
        mode=AGENT_MODE,
        publish_strategy=PUBLISH_STRATEGY,
        lookahead_depth=LOOKAHEAD_DEPTH,
        candidate_trials=CANDIDATE_TRIALS,
        branch_factor=BRANCH_FACTOR,
        max_expansions=MAX_EXPANSIONS,
        publish_horizon=PUBLISH_HORIZON,
        utility_weight=UTILITY_WEIGHT,
        logprob_weight=LOGPROB_WEIGHT,
        rollback_penalty=ROLLBACK_PENALTY,
        temperature=TEMPERATURE,
    )

RESUME_RUN = True
FORCE_REBUILD = False

sorted(CORPUS_REGISTRY)


## Run or resume

This creates a versioned run directory with:
- `run_manifest.json`
- `checkpoint_state.json`
- generation snapshots
- partial/final metrics CSVs
- saved figures with version tags and timestamps in the titles

In [ ]:
run = run_ngram_regeneration_lab(
    version=EXPERIMENT_VERSION,
    run_name=RUN_NAME,
    urtext_config=urtext_cfg,
    latent_config=latent_cfg,
    lab_config=lab_cfg,
    selection_config=selection_cfg,
    agent_config=agent_cfg,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    resume=RESUME_RUN,
    force_rebuild=FORCE_REBUILD,
    progress_bar=True,
)

results = pd.DataFrame(run.metrics_rows).sort_values("generation").reset_index(drop=True)

display(Markdown(f"**Run directory:** `{run.paths.run_dir}`"))
display(Markdown(f"**Checkpoint:** `{run.paths.checkpoint_path}`"))
display(Markdown(f"**Metrics CSV:** `{run.paths.metrics_final_path}`"))
display(Markdown(f"**Summary JSON:** `{run.paths.summary_path}`"))
display(Markdown(f"**Urtext summary:** `{run.manifest['source_summary']}`"))
display(Markdown(f"**Selection summary:** `{run.manifest['selection_summary']}`"))
display(Markdown(f"**Agent config:** `{run.manifest.get('agent_config')}`"))

results


In [ ]:
base_cols = [
    "generation",
    "vocab_size",
    "token_entropy_bits",
] + [f"distinct_{order}grams" for order in range(1, MAX_ORDER + 1)] + [
    f"distinct_{order}grams_ratio_vs_gen0" for order in range(1, MAX_ORDER + 1)
]

utility_cols = [
    col for col in [
        "utility_window_total",
        "desirable_windows",
        "neutral_windows",
        "undesirable_windows",
        "desirable_window_share",
        "neutral_window_share",
        "undesirable_window_share",
        "agent_decisions",
        "agent_desirable_rate",
        "agent_neutral_rate",
        "agent_undesirable_rate",
        "agent_avg_utility_score",
        "agent_avg_total_score",
        "agent_avg_search_cost",
    ]
    if col in results.columns
]

results[base_cols + utility_cols]


## Saved figures

All figures are saved to disk and can be redisplayed directly from the run directory.

In [ ]:
for figure_path in sorted(run.paths.figures_dir.glob("*.png")):
    display(Markdown(f"**{figure_path.name}**"))
    display(Image(filename=str(figure_path)))


In [ ]:
display(Markdown("**Saved sample texts**"))
print(run.paths.sample_text_path.read_text(encoding="utf-8"))


## Useful switches

Examples:
- theorem 1, synthetic latent 4-gram world: set `MAX_ORDER = 4`, extend `ORDER_KEEP_PROBS` to `{2: 0.5, 3: 0.25, 4: 0.125}`, keep `SELECTION_MODE = "none"`
- theorem 1, Jane Austen urtext: set `URTEXT_MODE = "author_corpus"`, `AUTHOR_KEY = "jane_austen"`
- theorem 2, dense implicit utility: set `SELECTION_MODE = "hash_rgram"`
- theorem 2, urtext reference 5-grams as desirable: set `SELECTION_MODE = "reference_rgram"`, `UTILITY_SPAN = 5`
- theorem 2, utility by frequency in the urtext: set `SELECTION_MODE = "reference_frequency_rgram"`
- publish whole selected lookahead blocks instead of one token at a time: increase `PUBLISH_HORIZON`
- penalize rollback away from greedy continuations: increase `ROLLBACK_PENALTY` and use `AGENT_MODE = "tree_search"`

Because the run state lives on disk, you can safely stop and resume as long as the version, run name, and configuration stay the same.